# Joined Time Series DataFrame

This notebook loads precomputed time series parameter CSVs (TSS, precipitation, discharge, wind, and LULC),
merges them into two final dataframes (`df_local` and `df_regional`) and writes the merged CSV outputs.

Purpose:
- Explain columns and data sources.
- Produce consolidated tables for statistical analysis.

Data sources (CSV files):
- `area_TSS_time_series.csv` : TSS statistics per period.
- `chirps_time_series.csv` : precipitation (local and regional means).
- `discharge_time_series.csv` : river discharge time series.
- `era5_time_series.csv` : wind components (local and regional).
- `lulc_data.csv` : land-use / land-cover areas by watershed and class.

Outputs:
- `df_merged_local.csv` and `df_merged_regional.csv` in the `datasets/Parameters Time series/merged_df` folder.

How to use:
1. Adjust paths below if your repo is in a different location.
2. Run the notebook from top to bottom to recreate merged CSVs.

In [2]:
# Load standard libraries
import pandas as pd
import os

## Load datasets
The next cells read CSV files created earlier in the workflow. Each file contains time-series summaries by sampling period.
Make sure the `datasets/Parameters Time series/Results` folder exists and contains the expected CSVs.

In [3]:
# Read TSS area statistics and keep only the columns we need
area_tss = pd.read_csv(r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\Results\area_TSS_time_series.csv')\
    .drop(columns=['Unnamed: 0','idx'])
area_tss = area_tss[['TSS_max', 'TSS_mean', 'TSS_min', 'TSS_stdDev', 'area_km2',
       'band_count', 'time_finish', 'time_start', 'water_period',
       'year']].copy()
area_tss.head()

,TSS_max,TSS_mean,TSS_min,TSS_stdDev,area_km2,band_count,time_finish,time_start,water_period,year
0,231.388351,48.812000,8.646607,27.405530,1043.404013,6,1985-04-09,1984-12-24,R,1984
1,230.723541,26.369015,7.906408,18.243428,1091.495113,6,1985-07-09,1985-04-09,HW,1985
2,233.251251,36.788005,8.202882,24.029145,885.768756,6,1985-09-23,1985-07-09,F,1985
3,234.163910,59.018131,10.828718,48.529756,981.618495,6,1985-12-16,1985-09-23,LW,1985
4,231.440628,20.104853,8.064194,22.183980,1131.200198,6,1986-08-23,1986-06-08,F,1986


In [4]:
# Read precipitation timeseries (CHIRPS) - contains local and regional means
precipitation = pd.read_csv(r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\Results\chirps_time_series.csv').drop(columns=['Unnamed: 0'])
precipitation.columns

Index(['band_count', 'mean_loc_chirps', 'mean_reg_chirps', 'month_init',
       'time_finish', 'time_start', 'water_period', 'year'],
      dtype='object')

In [5]:
# Read river discharge time series
discharge = pd.read_csv(r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\Results\discharge_time_series.csv').drop(columns=['Unnamed: 0'])
discharge.columns

Index(['water_period', 'time_start', 'time_finish', 'mean_discharge'], dtype='object')

In [6]:
# Read ERA5-derived wind components (u,v) with local and regional aggregates
wind = pd.read_csv(r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\Results\era5_time_series.csv').drop(columns=['Unnamed: 0'])
wind.columns

Index(['band_count', 'month_init', 'time_finish', 'time_start', 'u_wind_loc',
       'u_wind_reg', 'v_wind_loc', 'v_wind_reg', 'water_period', 'year'],
      dtype='object')

In [7]:
# Read land-use / land-cover area summaries by watershed and class
lulc = pd.read_csv(r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\Results\lulc_data.csv').drop(columns=['Unnamed: 0'])
lulc.columns

Index(['year', 'class_name', 'watershed', 'area_km2'], dtype='object')

# organize dataframes

## Local watershed

In [8]:
# Select local precipitation summary columns
precipitation_loc = precipitation[['mean_loc_chirps','time_finish', 'time_start', 'water_period', 'year']].copy()
precipitation_loc.head()

,mean_loc_chirps,time_finish,time_start,water_period,year
0,1051.117126,1985-04-09,1984-12-24,R,1984
1,578.634586,1985-07-09,1985-04-09,HW,1985
2,125.430803,1985-09-23,1985-07-09,F,1985
3,411.507501,1985-12-16,1985-09-23,LW,1985
4,912.281089,1986-03-16,1985-12-16,R,1985


In [9]:
# Select local wind components
wind_loc = wind[['time_finish', 'time_start', 'u_wind_loc',
        'v_wind_loc', 'water_period', 'year']].copy()
wind_loc.head()

,time_finish,time_start,u_wind_loc,v_wind_loc,water_period,year
0,1985-04-09,1984-12-24,-1.803958,-0.469250,R,1984
1,1985-07-09,1985-04-09,-2.042476,-0.087150,HW,1985
2,1985-09-23,1985-07-09,-1.711139,-0.183250,F,1985
3,1985-12-16,1985-09-23,-1.364044,-0.094174,LW,1985
4,1986-03-16,1985-12-16,-1.767657,-0.590797,R,1985


In [10]:
# Ensure time columns are proper datetimes and have a 'year' column for merging
list = [area_tss,
        precipitation_loc,
        discharge,
        wind_loc]

for df in list:
    df['time_finish'] = pd.to_datetime(df['time_finish'])
    df['time_start'] = pd.to_datetime(df['time_start'])
    df['year'] = df['time_start'].dt.year
    print(df.columns)

Index(['TSS_max', 'TSS_mean', 'TSS_min', 'TSS_stdDev', 'area_km2',
       'band_count', 'time_finish', 'time_start', 'water_period', 'year'],
      dtype='object')
Index(['mean_loc_chirps', 'time_finish', 'time_start', 'water_period', 'year'], dtype='object')
Index(['water_period', 'time_start', 'time_finish', 'mean_discharge', 'year'], dtype='object')
Index(['time_finish', 'time_start', 'u_wind_loc', 'v_wind_loc', 'water_period',
       'year'],
      dtype='object')


In [11]:
# Merge datasets to create a base dataframe, then add local precipitation and wind
df_base = area_tss.merge(discharge, on=['time_finish', 'time_start', 'water_period', 'year'],how='outer')
df_local = df_base.merge(precipitation_loc, on=['time_finish', 'time_start', 'water_period', 'year'],how='outer')
df_local = df_local.merge(wind_loc, on=['time_finish', 'time_start', 'water_period', 'year'],how='outer')
df_local

,TSS_max,TSS_mean,TSS_min,TSS_stdDev,area_km2,band_count,time_finish,time_start,water_period,year,mean_discharge,mean_loc_chirps,u_wind_loc,v_wind_loc
0,231.388351,48.812000,8.646607,27.405530,1043.404013,6.0,1985-04-09 00:00:00,1984-12-24 00:00:00,R,1984,169528.406475,1051.117126,-1.803958,-0.469250
1,230.723541,26.369015,7.906408,18.243428,1091.495113,6.0,1985-07-09 00:00:00,1985-04-09 00:00:00,HW,1985,NaN,578.634586,-2.042476,-0.087150
2,NaN,NaN,NaN,NaN,NaN,NaN,1985-07-09 06:00:00,1985-04-09 00:00:00,HW,1985,191042.140109,NaN,NaN,NaN
3,233.251251,36.788005,8.202882,24.029145,885.768756,6.0,1985-09-23 00:00:00,1985-07-09 00:00:00,F,1985,NaN,125.430803,-1.711139,-0.183250
4,NaN,NaN,NaN,NaN,NaN,NaN,1985-09-23 18:00:00,1985-07-09 06:00:00,F,1985,168948.841447,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,235.141968,126.397186,7.947257,86.057653,368.463271,6.0,2025-03-09 00:00:00,2024-11-23 00:00:00,R,2024,124708.110243,1205.653605,-1.683873,-0.251720
280,215.393143,21.700121,7.871317,13.777758,1303.141133,6.0,2025-06-09 00:00:00,2025-03-09 00:00:00,HW,2025,223941.292151,837.943684,-2.227315,-0.343642
281,215.301117,20.144170,7.920572,11.239461,1433.663327,6.0,2025-09-09 00:00:00,2025-06-09 00:00:00,F,2025,231944.084717,268.558520,-2.356343,-0.204407
282,231.284851,58.669482,8.254883,27.916471,1112.831699,6.0,2025-12-09 00:00:00,2025-09-09 00:00:00,LW,2025,NaN,218.504847,-2.075493,0.143591


## Regional watershed

In [12]:
# Select regional precipitation summary columns
precipitation_reg = precipitation[['mean_reg_chirps','time_finish', 'time_start', 'water_period', 'year']].copy()
precipitation_reg.head()

,mean_reg_chirps,time_finish,time_start,water_period,year
0,688.247514,1985-04-09,1984-12-24,R,1984
1,693.334073,1985-07-09,1985-04-09,HW,1985
2,350.424617,1985-09-23,1985-07-09,F,1985
3,404.135124,1985-12-16,1985-09-23,LW,1985
4,717.836152,1986-03-16,1985-12-16,R,1985


In [13]:
# Select regional wind components
wind_reg = wind[['time_finish', 'time_start', 'u_wind_reg',
        'v_wind_reg', 'water_period', 'year']].copy()
wind_reg.head()

,time_finish,time_start,u_wind_reg,v_wind_reg,water_period,year
0,1985-04-09,1984-12-24,-0.945100,-0.693335,R,1984
1,1985-07-09,1985-04-09,-0.727754,-0.126121,HW,1985
2,1985-09-23,1985-07-09,-0.577824,-0.238426,F,1985
3,1985-12-16,1985-09-23,-0.576755,-0.355455,LW,1985
4,1986-03-16,1985-12-16,-0.893920,-0.624207,R,1985


In [14]:
# Ensure time columns are proper datetimes and have a 'year' column for merging
list = [ precipitation_reg,
        wind_reg]

for df in list:
    df['time_finish'] = pd.to_datetime(df['time_finish'])
    df['time_start'] = pd.to_datetime(df['time_start'])
    df['year'] = df['time_start'].dt.year
    print(df.columns)

Index(['mean_reg_chirps', 'time_finish', 'time_start', 'water_period', 'year'], dtype='object')
Index(['time_finish', 'time_start', 'u_wind_reg', 'v_wind_reg', 'water_period',
       'year'],
      dtype='object')


In [15]:
# Build the regional dataframe by adding regional precipitation and wind
df_regional = df_base.merge(precipitation_reg, on=['time_finish', 'time_start', 'water_period', 'year'],how='outer')
df_regional = df_regional.merge(wind_reg, on=['time_finish', 'time_start', 'water_period', 'year'],how='outer')
df_regional

,TSS_max,TSS_mean,TSS_min,TSS_stdDev,area_km2,band_count,time_finish,time_start,water_period,year,mean_discharge,mean_reg_chirps,u_wind_reg,v_wind_reg
0,231.388351,48.812000,8.646607,27.405530,1043.404013,6.0,1985-04-09 00:00:00,1984-12-24 00:00:00,R,1984,169528.406475,688.247514,-0.945100,-0.693335
1,230.723541,26.369015,7.906408,18.243428,1091.495113,6.0,1985-07-09 00:00:00,1985-04-09 00:00:00,HW,1985,NaN,693.334073,-0.727754,-0.126121
2,NaN,NaN,NaN,NaN,NaN,NaN,1985-07-09 06:00:00,1985-04-09 00:00:00,HW,1985,191042.140109,NaN,NaN,NaN
3,233.251251,36.788005,8.202882,24.029145,885.768756,6.0,1985-09-23 00:00:00,1985-07-09 00:00:00,F,1985,NaN,350.424617,-0.577824,-0.238426
4,NaN,NaN,NaN,NaN,NaN,NaN,1985-09-23 18:00:00,1985-07-09 06:00:00,F,1985,168948.841447,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,235.141968,126.397186,7.947257,86.057653,368.463271,6.0,2025-03-09 00:00:00,2024-11-23 00:00:00,R,2024,124708.110243,898.932983,-0.812905,-0.467850
280,215.393143,21.700121,7.871317,13.777758,1303.141133,6.0,2025-06-09 00:00:00,2025-03-09 00:00:00,HW,2025,223941.292151,856.490467,-0.830971,-0.289036
281,215.301117,20.144170,7.920572,11.239461,1433.663327,6.0,2025-09-09 00:00:00,2025-06-09 00:00:00,F,2025,231944.084717,382.916565,-0.814023,-0.274272
282,231.284851,58.669482,8.254883,27.916471,1112.831699,6.0,2025-12-09 00:00:00,2025-09-09 00:00:00,LW,2025,NaN,311.053055,-0.731969,-0.141838


# include lulc data

## Local Watershed

In [16]:
# Extract LULC for the Local Watershed and split natural vs anthropogenic classes
lulc_local = lulc.loc[lulc['watershed'] == 'Local Watershed'].copy()
lulc_local_nat = lulc_local.loc[lulc_local['class_name'] == 'Natural'].copy()
lulc_local_nat = lulc_local_nat.rename(columns={'area_km2': 'natural_km2'}).drop(columns=['class_name'])
lulc_local_nat.head()

,year,watershed,natural_km2
2,1985,Local Watershed,1838.000682
6,1986,Local Watershed,1865.902793
10,1987,Local Watershed,1809.635155
14,1988,Local Watershed,1759.578980
18,1989,Local Watershed,1691.760531


In [17]:
# Prepare anthropogenic area column for the local watershed
lulc_local_ant = lulc_local.loc[lulc_local['class_name'] != 'Natural'].copy()
lulc_local_ant = lulc_local_ant.rename(columns={'area_km2': 'anthropogenic_km2'}).drop(columns=['class_name'])
lulc_local_ant.head()

,year,watershed,anthropogenic_km2
0,1985,Local Watershed,48.897800
4,1986,Local Watershed,49.877077
8,1987,Local Watershed,91.981528
12,1988,Local Watershed,93.984196
16,1989,Local Watershed,69.071330


In [18]:
lulc_local = lulc_local_ant.merge(lulc_local_nat, on=['year','watershed'], how='outer')
lulc_local.head()

,year,watershed,anthropogenic_km2,natural_km2
0,1985,Local Watershed,48.897800,1838.000682
1,1986,Local Watershed,49.877077,1865.902793
2,1987,Local Watershed,91.981528,1809.635155
3,1988,Local Watershed,93.984196,1759.578980
4,1989,Local Watershed,69.071330,1691.760531


In [19]:
lulc_local.tail()

,year,watershed,anthropogenic_km2,natural_km2
35,2020,Local Watershed,202.905644,1559.941249
36,2021,Local Watershed,196.595229,1521.152454
37,2022,Local Watershed,195.541853,1451.598098
38,2023,Local Watershed,199.570460,1626.906567
39,2024,Local Watershed,224.678151,1674.648490


In [20]:
# Attach LULC summaries to the local dataframe and add a watershed label
df_local = df_local.merge(lulc_local, on ="year", how='outer')
df_local['watershed'] = 'Local Watershed'
df_local.tail()

,TSS_max,TSS_mean,TSS_min,TSS_stdDev,area_km2,band_count,time_finish,time_start,water_period,year,mean_discharge,mean_loc_chirps,u_wind_loc,v_wind_loc,watershed,anthropogenic_km2,natural_km2
279,235.141968,126.397186,7.947257,86.057653,368.463271,6.0,2025-03-09,2024-11-23,R,2024,124708.110243,1205.653605,-1.683873,-0.251720,Local Watershed,224.678151,1674.64849
280,215.393143,21.700121,7.871317,13.777758,1303.141133,6.0,2025-06-09,2025-03-09,HW,2025,223941.292151,837.943684,-2.227315,-0.343642,Local Watershed,NaN,NaN
281,215.301117,20.144170,7.920572,11.239461,1433.663327,6.0,2025-09-09,2025-06-09,F,2025,231944.084717,268.558520,-2.356343,-0.204407,Local Watershed,NaN,NaN
282,231.284851,58.669482,8.254883,27.916471,1112.831699,6.0,2025-12-09,2025-09-09,LW,2025,NaN,218.504847,-2.075493,0.143591,Local Watershed,NaN,NaN
283,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-01,2025-12-09,R,2025,NaN,90.892970,NaN,NaN,Local Watershed,NaN,NaN


## Regional Watershed

In [21]:
lulc_regional = lulc.loc[lulc['watershed'] != 'Local Watershed'].copy()
lulc_regional_nat = lulc_regional.loc[lulc_regional['class_name'] == 'Natural'].copy()
lulc_regional_nat = lulc_regional_nat.rename(columns={'area_km2': 'natural_km2'}).drop(columns=['class_name'])
lulc_regional_nat.head()

,year,watershed,natural_km2
3,1985,Regional Watershed,305485.474607
7,1986,Regional Watershed,305474.722955
11,1987,Regional Watershed,304938.202241
15,1988,Regional Watershed,304585.406387
19,1989,Regional Watershed,304367.353637


In [22]:
lulc_regional_ant = lulc_regional.loc[lulc_regional['class_name'] != 'Natural'].copy()
lulc_regional_ant = lulc_regional_ant.rename(columns={'area_km2': 'anthropogenic_km2'}).drop(columns=['class_name'])
lulc_regional_ant.head()

,year,watershed,anthropogenic_km2
1,1985,Regional Watershed,1637.268807
5,1986,Regional Watershed,1567.067862
9,1987,Regional Watershed,1769.637364
13,1988,Regional Watershed,1819.227134
17,1989,Regional Watershed,1736.977802


In [23]:
lulc_regional = lulc_regional_ant.merge(lulc_regional_nat, on=['year','watershed'], how='outer')
lulc_regional.head()

,year,watershed,anthropogenic_km2,natural_km2
0,1985,Regional Watershed,1637.268807,305485.474607
1,1986,Regional Watershed,1567.067862,305474.722955
2,1987,Regional Watershed,1769.637364,304938.202241
3,1988,Regional Watershed,1819.227134,304585.406387
4,1989,Regional Watershed,1736.977802,304367.353637


In [24]:
# Attach LULC summaries to the regional dataframe and add a watershed label
df_regional = df_regional.merge(lulc_regional, on ="year", how='outer')
df_regional['watershed'] = 'Regional Watershed'
df_regional.tail()

,TSS_max,TSS_mean,TSS_min,TSS_stdDev,area_km2,band_count,time_finish,time_start,water_period,year,mean_discharge,mean_reg_chirps,u_wind_reg,v_wind_reg,watershed,anthropogenic_km2,natural_km2
279,235.141968,126.397186,7.947257,86.057653,368.463271,6.0,2025-03-09,2024-11-23,R,2024,124708.110243,898.932983,-0.812905,-0.467850,Regional Watershed,5247.887401,299740.774236
280,215.393143,21.700121,7.871317,13.777758,1303.141133,6.0,2025-06-09,2025-03-09,HW,2025,223941.292151,856.490467,-0.830971,-0.289036,Regional Watershed,NaN,NaN
281,215.301117,20.144170,7.920572,11.239461,1433.663327,6.0,2025-09-09,2025-06-09,F,2025,231944.084717,382.916565,-0.814023,-0.274272,Regional Watershed,NaN,NaN
282,231.284851,58.669482,8.254883,27.916471,1112.831699,6.0,2025-12-09,2025-09-09,LW,2025,NaN,311.053055,-0.731969,-0.141838,Regional Watershed,NaN,NaN
283,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-01,2025-12-09,R,2025,NaN,136.640289,NaN,NaN,Regional Watershed,NaN,NaN


In [25]:
df_local = df_local.dropna(subset=["TSS_mean"])
df_regional= df_regional.dropna(subset=["TSS_mean"])

In [28]:
df_regional.shape

(148, 17)

In [26]:
# Export merged dataframes to CSV for downstream analysis
exp_dir = r"C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\merged_df"

df_local.to_csv(os.path.join(exp_dir,'df_merged_local.csv'))
df_regional.to_csv(os.path.join(exp_dir,'df_merged_regional.csv'))